# Isharah-1000: Signer-Independent Continuous Saudi Sign Language Recognition (pose-based)

**Task:** map a sequence of 2D body/hand/face keypoints to the sequence of Arabic glosses that was signed
(continuous sign language recognition, CSLR). **Setting:** signer-independent, so the dev and test signers never appear in training.

This notebook is being rebuilt in two parts:

- **Part 1 (this version):** configuration, a smoke test of the Colab environment, and a data inspection section.
  It trains nothing. Its job is to check facts about the data before any modelling decision depends on them.
- **Part 2 (later):** preprocessing, models (BiLSTM-CTC baseline, Conformer-CTC), training, decoding, evaluation, inference helper.

## How to run

1. **Runtime:** in Colab, choose *Runtime > Change runtime type > T4 GPU*. Part 1 does not need the GPU, but the smoke test checks that it is there for Part 2.
2. **Run all cells top to bottom.** Every cell can be re-run. Copies to local disk are skipped if the file is already there.
3. **Data location:** the first folder in `CONFIG["DATA_DIR_CANDIDATES"]` that contains the train/dev pose file is used.
   If none does, the mount cell prints what to do (shortcut, own copy, or upload to `/content/data`).
4. **Checkpoints and logs** go to `CONFIG["CKPT_DIR"]` in *your own* My Drive, never inside the shared `SignLanguage` folder.
5. **After running, save the notebook** (Ctrl+S / Cmd+S) so the outputs are stored in the `.ipynb` file.
6. Everything in `CONFIG` marked **VERIFY AFTER INSPECTION** is an assumption. After Part 1 runs, it gets corrected from the printed outputs.

All paths in this notebook are on the Colab server, not on the laptop.

## 0. Configuration

**What:** one dictionary, `CONFIG`, that holds every path, data-layout assumption and (in Part 2) every hyperparameter.

**Why:** one place to change settings means experiments are reproducible and easy to compare, and nothing is hard-coded in later cells.
Values I could not know without seeing the data are marked **VERIFY AFTER INSPECTION**. Later cells check them with assertions
that stop the notebook with a clear message if they are wrong, instead of silently training on wrong inputs.

`FAST_DEV_RUN` is not used in Part 1. In Part 2 it switches the pipeline to a tiny subset and 1-2 epochs, to test everything end to end in minutes before a full run.

In [ ]:
CONFIG = {
    # ---- run mode (used in Part 2) ----
    "FAST_DEV_RUN": True,        # True: tiny subset + 1-2 epochs to test the full pipeline quickly
    "FAST_DEV_N_TRAIN": 256,     # training clips used when FAST_DEV_RUN is True
    "FAST_DEV_N_DEV": 64,        # dev clips used when FAST_DEV_RUN is True
    "FAST_DEV_EPOCHS": 2,
    "SEED": 42,

    # ---- paths on the Colab server ---- VERIFY AFTER INSPECTION
    "DRIVE_MOUNT": "/content/drive",
    # First folder that contains TRAIN_DEV_PKL is used.
    "DATA_DIR_CANDIDATES": [
        "/content/drive/MyDrive/SignLanguage/data",  # shared team folder (shortcut in My Drive)
        "/content/drive/MyDrive/isharah_data",       # fallback: your own copy in My Drive
        "/content/data",                             # fallback: files uploaded to the Colab server
    ],
    "SHARED_DIR": "/content/drive/MyDrive/SignLanguage",  # checkpoints must NOT go in here
    "LOCAL_DATA_DIR": "/content/data",                    # .pkl files are copied here before loading
    "TRAIN_DEV_PKL": "pose_data_isharah1000_hands_lips_body_May12.pkl",
    "TEST_PKL": "pose_data_isharah1000_SI_test.pkl",
    "PUBLIC_ZIP": "public_si_dat.zip",
    "OLD_CSV": {"train": "train.csv", "dev": "dev.csv"},
    "ANNOT_SUBDIR": "Annotations/SI",                     # official labels; use SI, never US
    "ANNOT_TXT": {"train": "train.txt", "dev": "dev.txt", "test": "test.txt"},
    "CKPT_DIR": "/content/drive/MyDrive/isharah_rebuild_ckpt",

    # ---- data layout ---- VERIFY AFTER INSPECTION
    "POSE_KEY": "keypoints",     # field inside each pkl entry that holds the (T, K, C) array
    "N_KEYPOINTS": 86,
    "N_COORDS": 2,
    # Filled in from the keypoint index plot in section 2.11, e.g. {"body": [...], "left_hand": [...], ...}.
    "KEYPOINT_GROUPS": None,
    "SHOULDER_IDX": None,        # (left_shoulder, right_shoulder) keypoint indices
    "LONG_CLIP_FRAMES": 300,     # report how many clips are longer than this
}

## 1. Smoke test (environment, Drive, paths, write access)

**What:** prints the GPU, library versions, RAM and disk; mounts Google Drive; finds the data folder; checks that every data file
exists and prints its size; writes and deletes a test file in `CKPT_DIR`.

**Why:** a long Colab run that fails after an hour because a path is wrong or Drive is read-only wastes the most time.
These checks fail in seconds instead. The write test matters because checkpoints are saved to Drive so training can resume after a disconnect.

`SignLanguage` is a shortcut to a folder shared with me. Colab only sees the Drive of the Google account it is signed in with,
so if that account is not the one the folder is shared with, the path is missing. The cell then prints the options to fix it.

In [ ]:
import csv, gc, json, os, pickle, platform, re, shutil, time, zipfile
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch


def print_environment():
    print(f"Python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name}, {p.total_memory / 1e9:.1f} GB, CUDA {torch.version.cuda}")
    else:
        print("WARNING: no GPU visible. Runtime > Change runtime type > T4 GPU (needed for Part 2).")
    vm = psutil.virtual_memory()
    print(f"RAM: {vm.total / 1e9:.1f} GB total, {vm.available / 1e9:.1f} GB available")
    du = shutil.disk_usage("/content" if os.path.isdir("/content") else "/")
    print(f"Local disk: {du.free / 1e9:.1f} GB free of {du.total / 1e9:.1f} GB")


print_environment()

In [ ]:
MISSING_DATA_HELP = "\n".join([
    "Could not find {pkl} in any folder of CONFIG['DATA_DIR_CANDIDATES'].",
    "SignLanguage is a shortcut to a shared folder. Colab only sees the Drive of the account it is signed in with.",
    "Fix it in one of these ways, then re-run this cell:",
    "  1. Sign in to drive.google.com with the account Colab uses (the one picked in the drive.mount prompt),",
    "     open 'Shared with me', right-click SignLanguage > Organize > Add shortcut > My Drive.",
    "  2. Copy the data folder into that account's My Drive as 'isharah_data' (same file names, keep Annotations/SI/).",
    "  3. Upload the files to /content/data on the Colab server (lost when the runtime is reset).",
    "  You can also add your own folder to the front of CONFIG['DATA_DIR_CANDIDATES'].",
])


def mount_drive(mount_point):
    if os.path.isdir(os.path.join(mount_point, "MyDrive")):
        print("Drive already mounted at", mount_point)
        return True
    try:
        from google.colab import drive
        drive.mount(mount_point)
        return True
    except Exception as e:  # not on Colab, or mount refused / not supported in this client
        print(f"Drive mount failed: {e!r}")
        print("If Drive cannot be mounted from this client, upload the files to /content/data (option 3 below).")
        return False


def resolve_data_dir(candidates, required_file):
    for d in candidates:
        found = os.path.isfile(os.path.join(d, required_file))
        print(f"  {'FOUND  ' if found else 'missing'} {d}")
        if found:
            return d
    print(MISSING_DATA_HELP.format(pkl=required_file))
    raise FileNotFoundError(f"{required_file} not found in any DATA_DIR candidate")


def build_paths(data_dir):
    annot = os.path.join(data_dir, CONFIG["ANNOT_SUBDIR"])
    paths = {
        "train_dev_pkl": os.path.join(data_dir, CONFIG["TRAIN_DEV_PKL"]),
        "test_pkl": os.path.join(data_dir, CONFIG["TEST_PKL"]),
        "public_zip": os.path.join(data_dir, CONFIG["PUBLIC_ZIP"]),
    }
    paths.update({f"csv_{s}": os.path.join(data_dir, f) for s, f in CONFIG["OLD_CSV"].items()})
    paths.update({f"txt_{s}": os.path.join(annot, f) for s, f in CONFIG["ANNOT_TXT"].items()})
    return paths


# The pipeline cannot run without these. The rest are checked and reported but optional.
REQUIRED_PATHS = ["train_dev_pkl", "test_pkl", "txt_train", "txt_dev"]


def check_paths(paths):
    for name, p in paths.items():
        if os.path.isfile(p):
            print(f"  OK       {name:14s} {os.path.getsize(p) / 1e6:10.1f} MB  {p}")
        else:
            print(f"  MISSING  {name:14s} {'':10s}     {p}")
    missing = [n for n in REQUIRED_PATHS if not os.path.isfile(paths[n])]
    assert not missing, f"Required files missing: {missing}. Fix CONFIG paths (VERIFY AFTER INSPECTION)."


mount_drive(CONFIG["DRIVE_MOUNT"])
print("Looking for data:")
DATA_DIR = resolve_data_dir(CONFIG["DATA_DIR_CANDIDATES"], CONFIG["TRAIN_DEV_PKL"])
PATHS = build_paths(DATA_DIR)
print("DATA_DIR =", DATA_DIR)
check_paths(PATHS)
annot_dir = os.path.join(DATA_DIR, CONFIG["ANNOT_SUBDIR"])
if os.path.isdir(annot_dir):
    print("Files in", annot_dir, ":", sorted(os.listdir(annot_dir)))

In [ ]:
def check_ckpt_dir(ckpt_dir, shared_dir):
    ckpt, shared = os.path.abspath(ckpt_dir), os.path.abspath(shared_dir)
    assert os.path.commonpath([ckpt, shared]) != shared, f"CKPT_DIR {ckpt} is inside the shared team folder {shared}"
    os.makedirs(ckpt, exist_ok=True)
    probe = os.path.join(ckpt, "_write_test.txt")
    with open(probe, "w") as f:
        f.write("ok")
    with open(probe) as f:
        assert f.read() == "ok", "read-back of the test file failed"
    os.remove(probe)
    assert not os.path.exists(probe), "test file could not be deleted"
    print("CKPT_DIR is writable:", ckpt, "| current contents:", sorted(os.listdir(ckpt)))


check_ckpt_dir(CONFIG["CKPT_DIR"], CONFIG["SHARED_DIR"])

## 2. Data inspection

**What:** loads the pose files and labels and prints facts about them: file structure, which IDs have labels and poses,
signers per split, clip lengths, missing keypoints, coordinate ranges, vocabulary and sentence overlap between splits,
and a plot of the keypoint indices.

**Why:** every later decision depends on these facts. Examples: which keypoint indices are the shoulders (needed for normalization),
whether coordinates are pixels or already in [0, 1], whether test labels exist, and how many dev/test glosses never appear in training
(these are errors the model cannot avoid). The old notebook skipped these checks, which is how problems such as building the vocabulary
from train + dev went unnoticed.

**Memory plan:** each `.pkl` is copied from Drive to the local disk first (reading from local disk is faster and does not depend on the
Drive connection), then loaded. Statistics are computed in one streaming pass per split, so no large concatenated array is built.
In section 2.5 everything later sections need from the train/dev poses is computed, the train/dev dict is deleted, and only then is the
test pose file loaded (for inspection only, then also deleted). RAM usage is printed before and after each delete.

**Train-only rule:** statistics of dev and test are printed here for reporting only. Every value the pipeline uses (maximum length,
outlier thresholds, vocabulary, normalization statistics) must be computed from the training split only. Choosing any of them from
dev or test data would leak information about unseen signers into the model.

**Missing keypoints:** a keypoint is counted as missing if it is NaN or exactly (0, 0), which is how pose extractors usually mark
"not detected". Section 2.8 shows whether that assumption fits this data.

The next cell only defines helper functions used by the inspection cells.

In [ ]:
STATS = {}  # split -> pose statistics, filled in sections 2.5 and 2.7


def get_kp(entry):
    # entries may be {"keypoints": array} or a bare array
    return np.asarray(entry[CONFIG["POSE_KEY"]] if isinstance(entry, dict) else entry)


def ram():
    vm = psutil.virtual_memory()
    rss = psutil.Process().memory_info().rss  # this notebook's own process
    return f"process {rss / 1e9:.2f} GB | system used {vm.used / 1e9:.1f} / {vm.total / 1e9:.1f} GB"


def copy_to_local(src, local_dir=None):
    local_dir = local_dir or CONFIG["LOCAL_DATA_DIR"]
    os.makedirs(local_dir, exist_ok=True)
    dst = os.path.join(local_dir, os.path.basename(src))
    if os.path.abspath(src) == os.path.abspath(dst):
        return dst
    size = os.path.getsize(src)
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        print("Local copy already present, skipping copy:", dst)
        return dst
    free = shutil.disk_usage(local_dir).free
    assert free > 1.2 * size, f"Not enough local disk: file {size / 1e9:.1f} GB, free {free / 1e9:.1f} GB"
    t0 = time.time()
    shutil.copyfile(src, dst)
    print(f"Copied {os.path.basename(src)} ({size / 1e9:.2f} GB) to {local_dir} in {time.time() - t0:.0f} s")
    return dst


def load_pickle(path):
    t0 = time.time()
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print(f"Loaded {os.path.basename(path)} in {time.time() - t0:.0f} s | {ram()}")
    return obj


def describe_pose_dict(poses, name):
    keys = list(poses.keys())
    print(f"[{name}] container={type(poses).__name__}, entries={len(keys)}, first keys={keys[:5]}")
    first = poses[keys[0]]
    if isinstance(first, dict):
        print("  fields in one entry:", {k: getattr(v, "shape", type(v).__name__) for k, v in first.items()})
    expected = (CONFIG["N_KEYPOINTS"], CONFIG["N_COORDS"])
    bad, empty, dtypes = [], [], Counter()
    for k in keys:
        a = get_kp(poses[k])
        dtypes[str(a.dtype)] += 1
        if a.ndim != 3 or a.shape[1:] != expected:
            bad.append((k, a.shape))
        elif a.shape[0] == 0:
            empty.append(k)
    print(f"  example shape={get_kp(first).shape}, dtypes={dict(dtypes)}, empty clips (T=0)={len(empty)} {empty[:5]}")
    assert not bad, (f"{len(bad)} entries are not (T, {expected[0]}, {expected[1]}), e.g. {bad[:5]}. "
                     "Fix POSE_KEY / N_KEYPOINTS / N_COORDS in CONFIG.")


def missing_mask(a):
    # (T, K, C) -> (T, K): True where the keypoint is NaN or exactly (0, 0)
    return np.isnan(a).any(-1) | (a == 0).all(-1)


def pose_stats(poses, ids, sample_per_clip=50, seed=0):
    # One streaming pass: clip lengths, missing rates, per-keypoint mean/std, coordinate range and a point sample for percentiles.
    K, C = CONFIG["N_KEYPOINTS"], CONFIG["N_COORDS"]
    rng = np.random.default_rng(seed)
    lengths = np.zeros(len(ids), dtype=np.int64)
    s, ss, n_valid, n_missing = np.zeros((K, C)), np.zeros((K, C)), np.zeros(K), np.zeros(K)
    n_nan, empty_frames = 0, 0
    lo, hi = np.full(C, np.inf), np.full(C, -np.inf)
    sample = []
    for j, i in enumerate(ids):
        a = get_kp(poses[i]).astype(np.float64)
        lengths[j] = len(a)
        if len(a) == 0:
            continue
        miss = missing_mask(a)
        n_nan += int(np.isnan(a).any(-1).sum())
        n_missing += miss.sum(0)
        empty_frames += int(miss.all(1).sum())
        valid = ~miss
        v = np.where(valid[..., None], a, 0.0)
        s += v.sum(0)
        ss += (v ** 2).sum(0)
        n_valid += valid.sum(0)
        pts = a[valid]
        if len(pts):
            lo, hi = np.minimum(lo, pts.min(0)), np.maximum(hi, pts.max(0))
            sample.append(pts[rng.choice(len(pts), min(sample_per_clip, len(pts)), replace=False)])
    n_frames = int(lengths.sum())
    mean = s / np.maximum(n_valid, 1)[:, None]
    std = np.sqrt(np.maximum(ss / np.maximum(n_valid, 1)[:, None] - mean ** 2, 0))
    return {"lengths": lengths, "n_frames": n_frames, "n_nan_points": n_nan, "empty_frames": empty_frames,
            "missing_rate": n_missing / max(n_frames, 1), "kp_mean": mean, "kp_std": std,
            "coord_min": lo, "coord_max": hi,
            "coord_sample": np.concatenate(sample) if sample else np.zeros((0, C))}


def pick_clean_frame(poses, ids, n_clips=50):
    best = None  # (id, frame index, missing count)
    for i in ids[:n_clips]:
        miss = missing_mask(get_kp(poses[i]).astype(np.float64)).sum(1)
        if len(miss) == 0:
            continue
        t = int(miss.argmin())
        if best is None or miss[t] < best[2]:
            best = (i, t, int(miss[t]))
    return best


def signer_of(sample_id):
    return sample_id.split("_")[0]  # IDs look like "SS_NNNN", SS = signer


def norm_gloss(g):
    return " ".join(g.split())

### 2.1 Train/dev pose file: structure

**What:** copies the train+dev `.pkl` to local disk, loads it, and prints its container type, number of entries, example keys,
the fields inside one entry, array shapes and dtypes. It asserts that every array is `(T, N_KEYPOINTS, N_COORDS)`.

**Why:** confirms the layout the rest of the code assumes. If the shape assertion fails, the config is wrong, not the data.

In [ ]:
train_dev_local = copy_to_local(PATHS["train_dev_pkl"])
train_dev_poses = load_pickle(train_dev_local)
describe_pose_dict(train_dev_poses, "train/dev pkl")
TRAIN_DEV_KEYS = set(train_dev_poses)

### 2.2 Labels: official SI annotations and older CSVs

**What:** reads `Annotations/SI/{train,dev,test}.txt` (pipe-separated `id|gloss|text`) and the older `train.csv` / `dev.csv` (`id,gloss`).
Prints row counts, columns and 3 examples per file, and whether the test file actually contains glosses.

**Why:** the official SI files are the labels we train and evaluate on. The older CSVs are loaded only to check that they agree.
The `text` column (Arabic translation) is kept but not used, because the task is gloss recognition. Whether test labels exist decides
whether Part 2 can report a test WER or only produce a submission file.

In [ ]:
def read_label_file(path):
    if path.endswith(".txt"):
        # QUOTE_NONE: Arabic text may contain quote characters that are not CSV quoting
        df = pd.read_csv(path, sep="|", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE, encoding="utf-8-sig")
    else:
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.apply(lambda col: col.str.strip())
    assert "id" in df.columns, f"{path}: no 'id' column; columns are {list(df.columns)}"
    return df


LABELS = {}  # source name ("txt/train", "csv/dev", ...) -> DataFrame
for split in CONFIG["ANNOT_TXT"]:
    p = PATHS[f"txt_{split}"]
    if os.path.isfile(p):
        LABELS[f"txt/{split}"] = read_label_file(p)
for split in CONFIG["OLD_CSV"]:
    p = PATHS[f"csv_{split}"]
    if os.path.isfile(p):
        LABELS[f"csv/{split}"] = read_label_file(p)

for name, df in LABELS.items():
    n_gloss = int((df["gloss"] != "").sum()) if "gloss" in df else 0
    print(f"\n=== {name}: {len(df)} rows | columns {list(df.columns)} | rows with a gloss: {n_gloss}")
    print(df.head(3).to_string(index=False))

t = LABELS.get("txt/test")
TEST_HAS_LABELS = t is not None and "gloss" in t and len(t) > 0 and bool((t["gloss"] != "").all())
print("\nTEST LABELS AVAILABLE:", TEST_HAS_LABELS)

### 2.3 Do label IDs match pose IDs?

**What:** for each label file, counts rows, duplicate IDs, IDs that have a pose sequence and IDs that do not (with examples).
Also lists pose entries that have no label, checks that no ID is in both train and dev, and compares the glosses of the official
`.txt` files with the older `.csv` files for the same IDs.

**Why:** a sample without a pose cannot be used, and a silent mismatch (for example different ID formats) would drop data without
any error. The split ID lists built here (`SPLIT_IDS`) are what Part 2 uses: official SI labels, restricted to IDs that have poses.

In [ ]:
def compare_ids(df, pose_keys, name, show=10):
    ids = df["id"].tolist()
    uniq = set(ids)
    no_pose = sorted(uniq - pose_keys)
    print(f"{name:10s} rows={len(ids):6d} unique={len(uniq):6d} duplicates={len(ids) - len(uniq):4d} "
          f"with pose={len(uniq & pose_keys):6d} without pose={len(no_pose):5d}")
    if no_pose:
        print("           e.g. without pose:", no_pose[:show])


def compare_sources(a, b, name_a, name_b, show=3):
    m = LABELS[a][["id", "gloss"]].merge(LABELS[b][["id", "gloss"]], on="id", how="outer", suffixes=("_a", "_b"), indicator=True)
    both = m[m["_merge"] == "both"]
    diff = both[both["gloss_a"].map(norm_gloss) != both["gloss_b"].map(norm_gloss)]
    print(f"{name_a} vs {name_b}: shared ids={len(both)}, only in {name_a}={int((m['_merge'] == 'left_only').sum())}, "
          f"only in {name_b}={int((m['_merge'] == 'right_only').sum())}, different gloss={len(diff)}")
    if len(diff):
        print(diff[["id", "gloss_a", "gloss_b"]].head(show).to_string(index=False))


for name in ["txt/train", "txt/dev", "csv/train", "csv/dev"]:
    if name in LABELS:
        compare_ids(LABELS[name], TRAIN_DEV_KEYS, name)

labelled = set().union(*(set(df["id"]) for n, df in LABELS.items() if n != "txt/test"))
unlabelled = sorted(TRAIN_DEV_KEYS - labelled)
print(f"\nTrain/dev pose entries with no train/dev label in any file: {len(unlabelled)}  e.g. {unlabelled[:10]}")

for src in ["txt", "csv"]:
    if f"{src}/train" in LABELS and f"{src}/dev" in LABELS:
        shared = set(LABELS[f"{src}/train"]["id"]) & set(LABELS[f"{src}/dev"]["id"])
        print(f"IDs in both {src}/train and {src}/dev: {len(shared)}  e.g. {sorted(shared)[:5]}")
for split in ["train", "dev"]:
    if f"txt/{split}" in LABELS and f"csv/{split}" in LABELS:
        compare_sources(f"txt/{split}", f"csv/{split}", f"txt/{split}", f"csv/{split}")

SPLIT_IDS = {s: sorted(set(LABELS[f"txt/{s}"]["id"]) & TRAIN_DEV_KEYS) for s in ["train", "dev"]}
print("\nUsable samples (official SI label + pose):", {s: len(v) for s, v in SPLIT_IDS.items()})
assert all(SPLIT_IDS.values()), "A split has no usable samples: check ID formats above."

### 2.4 Contents of `public_si_dat.zip`

**What:** lists the files inside the zip (count, total size, file types, first entries) without extracting it.

**Why:** we have not used this file before. It may contain annotations or pose data that duplicate or differ from the files we use,
so we should know what is in it.

In [ ]:
def list_zip(path, show=30):
    with zipfile.ZipFile(path) as z:
        infos = z.infolist()
    print(f"{len(infos)} entries, {sum(i.file_size for i in infos) / 1e9:.2f} GB uncompressed")
    print("top-level folders:", dict(Counter(i.filename.split("/")[0] for i in infos)))
    print("extensions:", dict(Counter(os.path.splitext(i.filename)[1] or "(none)" for i in infos)))
    for i in infos[:show]:
        print(f"  {i.file_size / 1e6:10.2f} MB  {i.filename}")


if os.path.isfile(PATHS["public_zip"]):
    list_zip(PATHS["public_zip"])
else:
    print("public_si_dat.zip not found at", PATHS["public_zip"])

### 2.5 Free the train/dev poses, then inspect the test pose file

**What:** the first cell computes the train and dev statistics used in 2.7-2.8 and picks the frame plotted in 2.11, then deletes the
train/dev pose dict. The second cell copies and loads the test `.pkl`, prints its structure, checks its IDs against `test.txt`,
checks that no test ID is also in the train/dev file, computes its statistics, then deletes it too. RAM is printed before and after each delete.

**Why:** Colab has about 12 GB of RAM, and holding both pose files at once is the easiest way to crash the runtime. Only small
statistics are kept. We still need to know the test data has the same layout before Part 2 relies on it. In Part 2 the test file is
loaded again only in the final test section. The test statistics are for reporting only (see the train-only rule above).

In [ ]:
# Compute everything later sections need from the train/dev poses, so the dict can be freed before the test file is loaded.
for split in ["train", "dev"]:
    STATS[split] = pose_stats(train_dev_poses, SPLIT_IDS[split])
PLOT_CLIP, PLOT_T, PLOT_MISSING = pick_clean_frame(train_dev_poses, SPLIT_IDS["train"])
PLOT_FRAME = get_kp(train_dev_poses[PLOT_CLIP])[PLOT_T].astype(np.float64)

print("Before deleting train/dev poses |", ram())
del train_dev_poses
gc.collect()
print("After deleting train/dev poses  |", ram())

In [ ]:
test_local = copy_to_local(PATHS["test_pkl"])
test_poses = load_pickle(test_local)
describe_pose_dict(test_poses, "test pkl")
TEST_KEYS = set(test_poses)
if "txt/test" in LABELS:
    compare_ids(LABELS["txt/test"], TEST_KEYS, "txt/test")
print("Test pose IDs also present in the train/dev pkl:", len(TEST_KEYS & TRAIN_DEV_KEYS))
SPLIT_IDS["test"] = sorted(TEST_KEYS & set(LABELS["txt/test"]["id"])) if TEST_HAS_LABELS else sorted(TEST_KEYS)
# Test statistics are for reporting only. Every config value (max length, outlier thresholds, vocab,
# normalization stats) must be computed from train only.
STATS["test"] = pose_stats(test_poses, SPLIT_IDS["test"])
print("Before deleting test poses |", ram())
del test_poses
gc.collect()
print("After deleting test poses  |", ram())

### 2.6 Signers per split

**What:** extracts the signer from each ID (`SS_NNNN`, SS = signer), prints clips per signer per split, and lists signers shared between splits.

**Why:** the task is signer-independent, so dev and test signers must not appear in training. If they did, the dev WER would be optimistic.
It also tells us how many different people the model learns from.

In [ ]:
id_pattern = re.compile(r"^\d+_\d+$")
for split, ids in SPLIT_IDS.items():
    odd = [i for i in ids if not id_pattern.match(i)]
    if odd:
        print(f"WARNING {split}: {len(odd)} IDs do not look like SS_NNNN, e.g. {odd[:5]}")

SIGNERS = {split: Counter(signer_of(i) for i in ids) for split, ids in SPLIT_IDS.items()}
print(pd.DataFrame(SIGNERS).fillna(0).astype(int).sort_index().to_string())
print({split: len(c) for split, c in SIGNERS.items()}, "signers per split")
for a, b in combinations(SIGNERS, 2):
    print(f"signers in both {a} and {b}: {sorted(set(SIGNERS[a]) & set(SIGNERS[b]))}")

### 2.7 Sequence lengths

**What:** frames per clip for each split: min, percentiles, mean, max, and how many clips are longer than `LONG_CLIP_FRAMES` (300), plus a histogram.

**Why:** lengths decide batch memory and how much temporal downsampling the model can use. CTC needs at least as many output frames
as target glosses, so very short clips limit the downsampling factor. The old notebook cropped clips to 300 frames;
this shows how many clips that would have cut.

In [ ]:
STATS = {s: STATS[s] for s in ["train", "dev", "test"]}  # fixed print order

LONG = CONFIG["LONG_CLIP_FRAMES"]
rows = {}
for split, st in STATS.items():
    L = st["lengths"]
    rows[split] = {"clips": len(L), "min": L.min(), "p5": np.percentile(L, 5), "median": np.median(L), "mean": L.mean(),
                   "p95": np.percentile(L, 95), "max": L.max(), f"> {LONG} frames": int((L > LONG).sum()), "total frames": st["n_frames"]}
print(pd.DataFrame(rows).T.to_string(float_format=lambda x: f"{x:.1f}"))

fig, ax = plt.subplots(figsize=(9, 4))
for split, st in STATS.items():
    ax.hist(st["lengths"], bins=60, alpha=0.5, label=split, density=True)
ax.axvline(LONG, color="k", ls="--", lw=1, label=f"{LONG} frames")
ax.set(xlabel="frames per clip", ylabel="density", title="Clip length distribution")
ax.legend()
plt.show()

### 2.8 Missing keypoints, coordinate ranges, per-keypoint variance

**What:** per split, how many keypoints are NaN or (0, 0), how many frames have no detected keypoint at all, and the range and percentiles
of the x and y values. Then a table with one row per keypoint index: mean position, standard deviation and missing rate.

**Why:**
- Missing rates tell us whether Part 2 must interpolate or mask keypoints. Hands are usually the parts that drop out, and they carry most of the signing.
- The coordinate range tells us whether values are pixels or normalized to [0, 1]. This matters for the normalization and for augmentation sizes.
- The per-keypoint table helps identify the body parts (points that barely move are face/body, points that move a lot are hands)
  and shows points with near-zero variance that carry no information.

In [ ]:
for split, st in STATS.items():
    n_pts = st["n_frames"] * CONFIG["N_KEYPOINTS"]
    print(f"{split:5s}: frames={st['n_frames']}, NaN points={st['n_nan_points']}, "
          f"missing points={st['missing_rate'].mean() * 100:.2f}% of {n_pts}, "
          f"frames with no keypoint={st['empty_frames']} ({st['empty_frames'] / max(st['n_frames'], 1) * 100:.2f}%)")

pct = [0.1, 1, 50, 99, 99.9]
rows = []
for split, st in STATS.items():
    smp = st["coord_sample"]
    for c, axis in enumerate("xy"[:CONFIG["N_COORDS"]]):
        r = {"split": split, "axis": axis, "min": st["coord_min"][c], "max": st["coord_max"][c]}
        r.update({f"p{p}": np.percentile(smp[:, c], p) for p in pct})
        r["share outside [0,1]"] = float(((smp[:, c] < 0) | (smp[:, c] > 1)).mean())
        rows.append(r)
coord_table = pd.DataFrame(rows)
print("\nCoordinate ranges (percentiles from a random sample of valid points):")
print(coord_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

lo, hi = coord_table["min"].min(), coord_table["max"].max()
if lo >= -0.05 and hi <= 1.05:
    print("=> values lie in [0, 1]: looks like normalized image coordinates")
elif hi > 2:
    print("=> values exceed 2: looks like pixel coordinates")
else:
    print("=> range is neither clearly [0, 1] nor pixels: check the table above")

tr = STATS["train"]
kp_table = pd.DataFrame({
    "mean_x": tr["kp_mean"][:, 0], "mean_y": tr["kp_mean"][:, 1],
    "std_x": tr["kp_std"][:, 0], "std_y": tr["kp_std"][:, 1],
    **{f"missing% {s}": STATS[s]["missing_rate"] * 100 for s in STATS},
})
kp_table.index.name = "kp"
print("\nPer-keypoint statistics (mean/std over train frames where the point is present):")
print(kp_table.to_string(float_format=lambda x: f"{x:.3f}"))

### 2.9 Gloss vocabulary overlap

**What:** builds the gloss vocabulary of each split (glosses are space-separated; multi-word concepts are joined with underscores),
and counts how many dev/test glosses never occur in training, both as distinct glosses and as a share of all gloss occurrences.

**Why:** the vocabulary must come from the training set only (the old notebook used train + dev, which leaks information).
A gloss that is not in the training vocabulary can never be predicted, so it is an unavoidable WER error. This count is a floor for the WER.

In [ ]:
SPLIT_LABELS = {s: LABELS[f"txt/{s}"].drop_duplicates("id").set_index("id").loc[SPLIT_IDS[s]] for s in ["train", "dev"]}
if TEST_HAS_LABELS:
    SPLIT_LABELS["test"] = LABELS["txt/test"].drop_duplicates("id").set_index("id").loc[SPLIT_IDS["test"]]

# Only VOCAB["train"] may become the model vocabulary; dev/test vocabularies are for reporting.
VOCAB = {s: Counter(tok for g in df["gloss"] for tok in g.split()) for s, df in SPLIT_LABELS.items()}
train_types = set(VOCAB["train"])
for s, v in VOCAB.items():
    print(f"{s:5s}: {len(v)} distinct glosses, {sum(v.values())} gloss occurrences, "
          f"{np.mean([len(g.split()) for g in SPLIT_LABELS[s]['gloss']]):.2f} glosses per sentence")
for s in [s for s in VOCAB if s != "train"]:
    unseen = set(VOCAB[s]) - train_types
    occ = sum(VOCAB[s][t] for t in unseen)
    print(f"{s}: {len(unseen)} distinct glosses not in train, {occ} occurrences "
          f"({occ / sum(VOCAB[s].values()) * 100:.2f}% of {s} gloss tokens). Examples: {sorted(unseen)[:10]}")
for a, b in combinations(VOCAB, 2):
    print(f"distinct glosses shared by {a} and {b}: {len(set(VOCAB[a]) & set(VOCAB[b]))}")

### 2.10 Exact-sentence overlap

**What:** for dev and test, how many samples have a gloss sentence that appears exactly (same glosses, same order) in the training set.

**Why:** all signers perform the same set of sentences, so dev/test sentences are probably all seen in training. Then the task is
"recognize a known sentence performed by a new person", not "recognize new sentences". That is easier, and it has to be reported
next to the final WER so the result is not misread. It also means a sentence-level language model would be very strong, which is
why Part 2 reports results with and without the language model.

In [ ]:
train_sents = set(SPLIT_LABELS["train"]["gloss"].map(norm_gloss))
print(f"train: {len(SPLIT_LABELS['train'])} samples, {len(train_sents)} distinct sentences")
for s in [s for s in SPLIT_LABELS if s != "train"]:
    sents = SPLIT_LABELS[s]["gloss"].map(norm_gloss)
    seen = sents.isin(train_sents)
    print(f"{s}: {len(sents)} samples, {sents.nunique()} distinct sentences | samples whose sentence is in train: "
          f"{int(seen.sum())} ({seen.mean() * 100:.2f}%) | distinct sentences not in train: {sents[~seen].nunique()}")

### 2.11 Keypoint index plot

**What:** takes the cleanest frame (fewest missing points) from the first training clips (picked in 2.5) and plots all keypoints with their index numbers,
coloured by index, so consecutive index ranges show as one colour. The y axis is flipped so the image is upright. The frame's coordinates
are also printed as a table.

**Why:** the pkl does not say which index is which body part. From this plot we fill `KEYPOINT_GROUPS` and `SHOULDER_IDX` in the config.
The shoulders are needed for the body-centred normalization in Part 2 (centre on the shoulder midpoint, scale by shoulder width),
and the groups let us test whether, for example, face points help or hurt for unseen signers.

In [ ]:
def plot_keypoint_indices(frame, title):
    idx = np.arange(len(frame))
    ok = ~missing_mask(frame[None])[0]
    fig, ax = plt.subplots(figsize=(14, 14))
    sc = ax.scatter(frame[ok, 0], frame[ok, 1], c=idx[ok], cmap="turbo", s=30)
    for k in idx[ok]:
        ax.annotate(str(k), frame[k], fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.invert_yaxis()  # image coordinates: y grows downwards
    ax.set_aspect("equal")
    ax.set_title(title)
    fig.colorbar(sc, ax=ax, shrink=0.6, label="keypoint index")
    plt.show()
    return idx[~ok]


missing_in_frame = plot_keypoint_indices(PLOT_FRAME, f"clip {PLOT_CLIP}, frame {PLOT_T} ({PLOT_MISSING} missing points)")
print("Missing keypoint indices in this frame:", missing_in_frame.tolist())
print(pd.DataFrame(PLOT_FRAME, columns=["x", "y"][:CONFIG["N_COORDS"]]).rename_axis("kp").to_string(float_format=lambda x: f"{x:.4f}"))

### 2.12 Config checks and inspection summary

**What:** validates the keypoint settings in `CONFIG` (once they are filled in, groups must cover every index exactly once and the
shoulders must be body points), and saves a short summary of the inspection to `CKPT_DIR/inspection_summary.json`.

**Why:** the validation is what makes a wrong `KEYPOINT_GROUPS` or `SHOULDER_IDX` fail loudly in Part 2 instead of producing a
quietly worse model. The summary file keeps the facts the pipeline is built on next to the checkpoints.

In [ ]:
def validate_keypoint_config(cfg):
    groups, shoulders, K = cfg["KEYPOINT_GROUPS"], cfg["SHOULDER_IDX"], cfg["N_KEYPOINTS"]
    if groups is None or shoulders is None:
        print("KEYPOINT_GROUPS / SHOULDER_IDX not set yet: fill them from the plot in 2.11 before Part 2.")
        return False
    all_idx = [i for g in groups.values() for i in g]
    assert len(all_idx) == len(set(all_idx)), "KEYPOINT_GROUPS: an index is in more than one group"
    assert set(all_idx) == set(range(K)), f"KEYPOINT_GROUPS must cover 0..{K - 1}; missing {sorted(set(range(K)) - set(all_idx))}"
    assert "body" in groups and all(s in groups["body"] for s in shoulders), "SHOULDER_IDX must be indices in the 'body' group"
    print("Keypoint config OK:", {g: len(v) for g, v in groups.items()}, "| shoulders:", shoulders)
    return True


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, np.generic):
        return o.item()
    return o


validate_keypoint_config(CONFIG)
SUMMARY = {
    "data_dir": DATA_DIR,
    "pkl_entries": {"train_dev": len(TRAIN_DEV_KEYS), "test": len(TEST_KEYS)},
    "label_rows": {n: len(df) for n, df in LABELS.items()},
    "usable_samples": {s: len(v) for s, v in SPLIT_IDS.items()},
    "test_has_labels": TEST_HAS_LABELS,
    "signers": {s: sorted(c) for s, c in SIGNERS.items()},
    "length": {s: {"min": st["lengths"].min(), "median": np.median(st["lengths"]), "max": st["lengths"].max(),
                   f"over_{LONG}": int((st["lengths"] > LONG).sum())} for s, st in STATS.items()},
    "missing_point_pct": {s: st["missing_rate"].mean() * 100 for s, st in STATS.items()},
    "coord_min": {s: st["coord_min"] for s, st in STATS.items()},
    "coord_max": {s: st["coord_max"] for s, st in STATS.items()},
    "vocab_size": {s: len(v) for s, v in VOCAB.items()},
    "unseen_glosses": {s: len(set(VOCAB[s]) - train_types) for s in VOCAB if s != "train"},
}
summary_path = os.path.join(CONFIG["CKPT_DIR"], "inspection_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(to_jsonable(SUMMARY), f, ensure_ascii=False, indent=1)
print(json.dumps(to_jsonable(SUMMARY), ensure_ascii=False, indent=1))
print("Saved", summary_path)

### Findings (fill in after running)

Write down what the outputs above showed, using only the printed numbers:

- Test labels available: ...
- Usable samples per split: ...
- Signers per split / any overlap: ...
- Coordinates are pixels or normalized: ...
- Missing keypoints (which indices, how often): ...
- Keypoint groups and shoulder indices (from 2.11): ...
- Dev/test glosses unseen in train, sentence overlap: ...